In [1]:
import os
import random
import gc

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

from scipy.stats import skew, mode
from tqdm import tqdm

import cv2
import pydicom

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

SEED = 1337


def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)




In [2]:
df_train = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/train.csv")
df_test = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/test.csv")
df_submission = pd.read_csv(
    "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)

print(
    f'Training Set Shape = {df_train.shape} - Patients = {df_train["Patient"].nunique()}'
)
print(f'Test Set Shape = {df_test.shape} - Patients = {df_test["Patient"].nunique()}')
print(f"Sample Submission Shape = {df_submission.shape}")




Training Set Shape = (1394, 7) - Patients = 158
Test Set Shape = (18, 7) - Patients = 18
Sample Submission Shape = (1908, 3)


In [3]:
class Preprocessor:
    def __init__(
        self, df_train, df_test, df_submission, n_folds, shuffle, resize_shape
    ):
        self.df_train = df_train.copy()
        self.df_train.sort_values(by=["Patient", "Weeks"], inplace=True)
        self.df_test = df_test.copy()
        self.df_submission = df_submission.copy()
        self.n_folds = n_folds
        self.shuffle = shuffle
        self.resize_shape = resize_shape

    def _drop_duplicates(self):
        self.df_train["FVC"] = self.df_train.groupby(["Patient", "Weeks"])[
            "FVC"
        ].transform("mean")
        self.df_train["Percent"] = self.df_train.groupby(["Patient", "Weeks"])[
            "Percent"
        ].transform("mean")
        self.df_train.drop_duplicates(inplace=True)
        self.df_train.reset_index(drop=True, inplace=True)

    def _label_encode(self):
        for df in [self.df_train, self.df_test]:
            df["Sex"] = df["Sex"].map({"Male": 0, "Female": 1})
            df["SmokingStatus"] = df["SmokingStatus"].map(
                {"Never smoked": 0, "Ex-smoker": 1, "Currently smokes": 2}
            )

    def _create_folds(self):
        patients = self.df_train["Patient"].unique()
        if self.shuffle:
            np.random.seed(SEED)
            np.random.shuffle(patients)
        folds = np.array_split(patients, self.n_folds)
        for fold_idx, patient_group in enumerate(folds, 1):
            self.df_train.loc[
                self.df_train["Patient"].isin(patient_group), "CV1_Fold"
            ] = fold_idx
            self.df_train.loc[
                self.df_train["Patient"].isin(patient_group), "CV2_Fold"
            ] = fold_idx
            self.df_train.loc[
                self.df_train["Patient"].isin(patient_group), "CV3_Fold"
            ] = fold_idx

    def _create_baseline_features(self):
        self.df_submission["Type"] = "Test"
        self.df_submission["Patient"] = self.df_submission["Patient_Week"].apply(
            lambda x: x.split("_")[0]
        )
        self.df_submission["Weeks"] = self.df_submission["Patient_Week"].apply(
            lambda x: int(x.split("_")[1])
        )
        self.df_submission.drop(
            columns=["Patient_Week", "FVC", "Confidence"], inplace=True
        )

        self.df_train["Type"] = "Train"
        self.df_train["Weeks_Passed"] = self.df_train["Weeks"] - self.df_train.groupby(
            "Patient"
        )["Weeks"].transform("min")
        self.df_train["FVC_Baseline"] = self.df_train.groupby("Patient")[
            "FVC"
        ].transform("first")

        for patient in self.df_test["Patient"].unique():
            mask = self.df_submission["Patient"] == patient
            self.df_submission.loc[mask, "FVC_Baseline"] = self.df_test.loc[
                self.df_test["Patient"] == patient, "FVC"
            ].values[0]
            self.df_submission.loc[mask, "Percent"] = self.df_test.loc[
                self.df_test["Patient"] == patient, "Percent"
            ].values[0]
            self.df_submission.loc[mask, "Age"] = self.df_test.loc[
                self.df_test["Patient"] == patient, "Age"
            ].values[0]
            self.df_submission.loc[mask, "Sex"] = self.df_test.loc[
                self.df_test["Patient"] == patient, "Sex"
            ].values[0]
            self.df_submission.loc[mask, "SmokingStatus"] = self.df_test.loc[
                self.df_test["Patient"] == patient, "SmokingStatus"
            ].values[0]

        self.df_submission["Weeks_Passed"] = self.df_submission["Weeks"]

        self.df_all = pd.concat([self.df_train, self.df_submission], ignore_index=True)
        self.df_all["Age"] = self.df_all["Age"] + np.floor(
            self.df_all["Weeks_Passed"] / 52
        ).astype(np.int8)

        for col in ["Weeks", "Weeks_Passed"]:
            self.df_all[col] = self.df_all[col].astype(np.int16)
        self.df_all["Age"] = self.df_all["Age"].astype(np.float32)
        self.df_all["FVC_Baseline"] = self.df_all["FVC_Baseline"].astype(np.float32)
        self.df_all["Percent"] = self.df_all["Percent"].astype(np.float32)
        self.df_all["Sex"] = self.df_all["Sex"].astype(np.uint8)
        self.df_all["SmokingStatus"] = self.df_all["SmokingStatus"].astype(np.uint8)

        self.df_train = self.df_all[self.df_all["Type"] == "Train"].drop(
            columns=["Type"]
        )
        for i in range(1, 4):
            self.df_train[f"CV{i}_Fold"] = self.df_train[f"CV{i}_Fold"].astype(np.uint8)
        self.df_test = self.df_all[self.df_all["Type"] == "Test"].drop(
            columns=["Type", "FVC", f"CV1_Fold", f"CV2_Fold", f"CV3_Fold"]
        )

    def get_data(self):
        self._drop_duplicates()
        self._label_encode()
        self._create_folds()
        self._create_baseline_features()
        print(f"Preprocessed Training Shape = {self.df_train.shape}")
        print(f"Preprocessed Test Shape = {self.df_test.shape}")
        return self.df_train.copy(), self.df_test.copy()




In [4]:
seed_everything(SEED)

preprocessor_params = {
    "df_train": df_train,
    "df_test": df_test,
    "df_submission": df_submission,
    "n_folds": 2,
    "shuffle": True,
    "resize_shape": (
        512,
        512,
    ),  # kept for compatibility; not used in this simplified pipeline
}
preproc = Preprocessor(**preprocessor_params)
df_train, df_test = preproc.get_data()




Preprocessed Training Shape = (1387, 12)
Preprocessed Test Shape = (1908, 8)


In [5]:
predictors = [
    "Age",
    "Sex",
    "SmokingStatus",
    "FVC_Baseline",
    "Percent",
    "Weeks_Passed",
    "Weeks",  # new feature
]

X_train = df_train[predictors]
y_train = df_train["FVC"]

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof_preds = np.zeros(len(X_train))
for train_idx, val_idx in kf.split(X_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    model = GradientBoostingRegressor(
        random_state=SEED,
        n_estimators=3000,  # increased capacity
        learning_rate=0.05,
        max_depth=5,  # slightly deeper tree
    )
    model.fit(X_tr, y_tr)
    oof_preds[val_idx] = model.predict(X_val)

residuals = np.abs(y_train - oof_preds)

conf_model = GradientBoostingRegressor(
    random_state=SEED,
    n_estimators=1500,  # more trees for confidence estimation
    learning_rate=0.05,
    max_depth=3,
)
conf_model.fit(X_train, residuals)

test_confidence = conf_model.predict(df_test[predictors])
# Increase scaling factor to 5.0 to raise σ modestly and improve the Laplace‑Log‑Likelihood score
test_confidence = np.maximum(70, test_confidence * 5.0)

gbr = GradientBoostingRegressor(
    random_state=SEED, n_estimators=3000, learning_rate=0.05, max_depth=5
)
gbr.fit(X_train, y_train)

df_test["FVC"] = gbr.predict(df_test[predictors])
df_test["Confidence"] = test_confidence




In [6]:
submission = df_test[["Patient", "Weeks"]].copy()
submission["Patient_Week"] = (
    submission["Patient"].astype(str) + "_" + submission["Weeks"].astype(str)
)
submission["FVC"] = df_test["FVC"]
submission["Confidence"] = df_test["Confidence"]

submission[["Patient_Week", "FVC", "Confidence"]].to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")

Submission saved to submission.csv
